# J3: graph model on the "humanise a cycle" task

Trains the graph model on leave-one-take-out folds (V0 = no symmetry, V1 = beat symmetry C4, V3 = full rotation C12; V4 = V0 + helix edge + correction to B1; V5 = V0 + helix edge only) and compares it with the reference models B1 and B2 on the same strokes. Metrics and settings go to your Weights & Biases project; no corpus data leaves this session.

**Before running:** if OSF refuses the download (it sometimes answers 403), put a copy of the annotations in your Drive as `djembe_corpus.zip` (made on your computer, see the repository's README); the notebook then never contacts OSF. In Colab's left bar, open **Secrets** (key icon), add `WANDB_API_KEY` with your key from https://wandb.ai/authorize, and turn on *Notebook access*. Then *Runtime > Change runtime type > GPU* (optional, the model is small) and *Run all*.

In [ ]:
# Settings
REPO = "https://github.com/clementdumeril/Djembe.git"
REF = "main"                      # pin a commit hash here to reproduce a given run exactly
WANDB_PROJECT = "djembe"          # set to None to run without Weights & Biases
VARIANTS = "V0 V5"             # V4 (correction to B1) was dropped; the J3 runs: "V0 V1 V3" with SELECT = "best"
SEEDS = "0 1 2"
EPOCHS = 40
SELECT = "last"                # weights scored: final epoch ("last") or best validation epoch ("best")
TAG = "j3b"                    # W&B group prefix and results file name
USE_DRIVE_CACHE = True     # keep a private copy of the annotations in your Drive (OSF is sometimes unavailable)

In [ ]:
import os, subprocess, sys, torch


def sh(*cmd, **env):
    """Run a command, stream its output, and stop the notebook if it fails."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, **env})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {' '.join(cmd[:4])} ... - see its output above.")


print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is fine, slower)")
if not os.path.exists("/content/Djembe"):
    sh("git", "clone", "-q", REPO, "/content/Djembe")
sh("git", "-C", "/content/Djembe", "fetch", "-q", "origin")   # a clone left by an earlier session gets the new code
sh("git", "-C", "/content/Djembe", "checkout", "-q", "--detach", f"origin/{REF}" if REF == "main" else REF)
os.chdir("/content/Djembe")
sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".", "wandb", "pytest")
sh("git", "log", "--oneline", "-1")

In [ ]:
# Annotations only (about 7 MB, CC BY-NC-ND 4.0): a private copy for your own use, never redistributed.
import shutil, zipfile
CACHE = "/content/drive/MyDrive/djembe_corpus"
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.exists(CACHE + ".zip"):
        zipfile.ZipFile(CACHE + ".zip").extractall("/content/corpus")
        print("annotations unzipped from your Drive file djembe_corpus.zip")
    if os.path.isdir(CACHE):
        shutil.copytree(CACHE, "/content/corpus", dirs_exist_ok=True)
        print("annotations copied from your Drive folder djembe_corpus")
# downloads only what is missing; does not contact OSF at all if nothing is
sh(sys.executable, "-m", "djembe.dataset", "download", "--corpus", "/content/corpus")
if USE_DRIVE_CACHE:   # complete copy only: the download step raises if anything is missing
    shutil.copytree("/content/corpus", CACHE, dirs_exist_ok=True)
sh(sys.executable, "-m", "djembe.dataset", "build", "--corpus", "/content/corpus")
sh(sys.executable, "-m", "pytest", "-q", "tests", DJEMBE_CORPUS="/content/corpus")

In [ ]:
if WANDB_PROJECT:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")   # read from Colab Secrets, never printed
    import wandb
    wandb.login()


In [ ]:
args = ["--variants", *VARIANTS.split(), "--seeds", *SEEDS.split(), "--epochs", str(EPOCHS),
        "--select", SELECT, "--group-prefix", TAG, "--out", f"results/{TAG}_humanize.json"]
if WANDB_PROJECT:
    args += ["--wandb-project", WANDB_PROJECT]
sh(sys.executable, "experiments/gnn/train_humanize.py", *args)

In [ ]:
import json, os
assert os.path.exists(f"results/{TAG}_humanize.json"), "No results yet: the training cell above did not finish."
s = json.load(open(f"results/{TAG}_humanize.json"))["summary"]
for name, v in s.items():
    print(f"{name:16s} offsets {v['nll_delta']:+.3f}   intensity {v['nll_intensity']:.3f}   ({v['runs']} runs)")
if USE_DRIVE_CACHE:   # results only (metrics, no corpus data): kept in your Drive, safe from session resets
    os.makedirs("/content/drive/MyDrive/djembe_results", exist_ok=True)
    shutil.copy(f"results/{TAG}_humanize.json", f"/content/drive/MyDrive/djembe_results/{TAG}_humanize.json")
    print(f"saved to Drive: djembe_results/{TAG}_humanize.json")
